# Calendar Spread IV Crush Calculator, step by step

> **Disclaimer.** Educational and research use only. This is not investment advice.

## What this calculator is for

Before an earnings release, options that expire just after the event carry extra *implied volatility* (IV),
because the market prices in a big move. Once earnings are out, that extra IV collapses. This is the **IV crush**.

A **calendar spread** sells the near-dated (front) option and buys a later-dated (back) option at the same strike.
It makes money when the front option's IV falls more than the back option's IV, which is exactly what an IV crush does.

The calculator screens a stock for whether conditions favour this trade, using three checks:

| Check | What it measures | Pass when |
|---|---|---|
| `avg_volume` | Average daily share volume over 30 days: is the stock liquid enough? | ≥ 1,500,000 |
| `iv30_rv30` | 30-day implied vol ÷ 30-day realised vol: are options expensive relative to how much the stock actually moves? | ≥ 1.25 |
| `ts_slope_0_45` | How fast IV falls from the front expiry out to 45 days: is the front month inflated? | ≤ −0.00406 |

**Verdict.** All three pass → **Recommended**. The slope passes and exactly one of the other two passes → **Consider**. Otherwise → **Avoid**.

It also reports the **expected move**: the price of the front at-the-money straddle as a percentage of the stock price.

## How this notebook is organised

Each section defines **one small function** and then calls it on real data, so you can see every intermediate value.
The last section joins the pieces into `evaluate()`, the single function the rest of the code calls.
That is the same structure as `src/calculators/calendar_iv_crush.py` in chrysos-lab.

The notebook is self-contained. It doesn't import anything from the repo, so it runs on Google Colab as-is.

## 0. Setup

Colab already has `numpy`, `pandas` and `matplotlib`. Only `yfinance` (free Yahoo Finance data) needs installing.

In [ ]:
%pip install -q yfinance

In [ ]:
import math
from dataclasses import dataclass, field
from datetime import date, datetime, timedelta
from typing import Literal

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yfinance as yf

pd.set_option("display.width", 120)


class CalculatorError(ValueError):
    """Raised when the calculator can't produce a result from the data it was given."""

### Choose a ticker and a data source

- `SYMBOL`: the stock to screen.
- `DATA_SOURCE`:
  - `"yahoo"` uses live Yahoo Finance data.
  - `"synthetic"` generates realistic fake data, so the notebook runs at any hour.

**Timing matters for `"yahoo"`.** US stock options trade only from 9:30 AM to 4:00 PM New York time.
After the close, Yahoo keeps serving the last quotes for a while. Overnight it clears them: bid/ask become 0 and IV drops to a placeholder of 0.00001.
If you run during that window, Step 6 stops with a "stale quotes" error. Switch to `"synthetic"` or try again after the US open.

In [ ]:
SYMBOL = "NFLX"
DATA_SOURCE = "yahoo"  # "yahoo" or "synthetic"
AS_OF = date.today()  # the date that days-to-expiry (DTE) is counted from

## 1. Parameters

All thresholds live in one frozen dataclass rather than being scattered through the code.
That makes them easy to tweak, for example a stricter `min_iv30_rv30` for your own playbook, without touching the logic.

- `max_dte = 45` is the far end of the term-structure slope. It also decides which expiries are fetched (Step 3).
- `window = 30` is the lookback used for IV30, RV30 and average volume.

In [ ]:
@dataclass(frozen=True)
class Params:
    min_avg_volume: float = 1_500_000
    min_iv30_rv30: float = 1.25
    max_ts_slope_0_45: float = -0.00406
    max_dte: int = 45
    window: int = 30
    history_period: str = "3mo"


params = Params()
params

## 2. Data providers

The calculator never talks to Yahoo directly. It asks a **provider** for four things:

| Method | Returns |
|---|---|
| `expirations(symbol)` | all listed option expiry dates |
| `option_chain(symbol, expiry)` | calls and puts for one expiry, with columns `strike, iv, bid, ask` |
| `spot(symbol)` | the latest stock price |
| `price_history(symbol, period)` | daily bars with `open, high, low, close, volume` |

Every provider returns the **same column names**, whatever the source calls them. Yahoo uses
`impliedVolatility` and `Close`, so the Yahoo provider renames them to `iv` and `close`.
Because of this, the maths below works unchanged on Yahoo data, synthetic data, or later moomoo or a database cache.

In [ ]:
@dataclass(frozen=True)
class OptionChain:
    expiry: date
    calls: pd.DataFrame  # columns: strike, iv, bid, ask
    puts: pd.DataFrame


CHAIN_COLUMNS = ["strike", "iv", "bid", "ask"]
PRICE_COLUMNS = ["open", "high", "low", "close", "volume"]


class YFinanceProvider:
    def __init__(self):
        self._tickers = {}

    def _ticker(self, symbol):
        if symbol not in self._tickers:
            self._tickers[symbol] = yf.Ticker(symbol)
        return self._tickers[symbol]

    def expirations(self, symbol):
        try:
            raw = self._ticker(symbol).options
        except Exception:
            return []
        return [datetime.strptime(d, "%Y-%m-%d").date() for d in raw]

    def option_chain(self, symbol, expiry):
        chain = self._ticker(symbol).option_chain(expiry.strftime("%Y-%m-%d"))
        normalise = lambda df: df.rename(columns={"impliedVolatility": "iv"})[CHAIN_COLUMNS]
        return OptionChain(expiry, normalise(chain.calls), normalise(chain.puts))

    def spot(self, symbol):
        today = self._ticker(symbol).history(period="1d")
        return None if today.empty else float(today["Close"].iloc[-1])

    def price_history(self, symbol, period="3mo"):
        history = self._ticker(symbol).history(period=period).rename(columns=str.lower)
        return history[PRICE_COLUMNS]

The synthetic provider builds a plausible pre-earnings setup.
Front-month IV is inflated and decays with DTE, option prices follow a rough at-the-money approximation, and the price history is a seeded random walk.
It is only there so you can follow every step when Yahoo has no live quotes.

In [ ]:
class SyntheticProvider:
    def __init__(self, as_of, spot=100.0, seed=7):
        self.as_of, self._spot, self._seed = as_of, spot, seed

    def expirations(self, symbol):
        return [self.as_of + timedelta(days=d) for d in (0, 3, 10, 17, 24, 31, 38, 59, 73)]

    def option_chain(self, symbol, expiry):
        dte = max((expiry - self.as_of).days, 1)
        atm_iv = 0.35 + 0.9 * math.exp(-dte / 6)  # inflated front month, decaying to ~0.35
        strikes = np.arange(self._spot * 0.8, self._spot * 1.2 + 0.01, 2.5)
        t = dte / 365
        time_value = 0.4 * self._spot * atm_iv * math.sqrt(t) * np.exp(-np.abs(strikes - self._spot) / (0.1 * self._spot))

        def side(intrinsic):
            mid = intrinsic + time_value
            return pd.DataFrame({"strike": strikes, "iv": atm_iv + 0.02 * np.abs(strikes - self._spot) / 10,
                                 "bid": mid * 0.97, "ask": mid * 1.03})

        return OptionChain(expiry, side(np.maximum(self._spot - strikes, 0)), side(np.maximum(strikes - self._spot, 0)))

    def spot(self, symbol):
        return self._spot

    def price_history(self, symbol, period="3mo"):
        days = 63
        rng = np.random.default_rng(self._seed)  # same bars on every call
        close = self._spot * np.exp(np.cumsum(rng.normal(0, 0.018, days)))
        close *= self._spot / close[-1]
        open_ = close * np.exp(rng.normal(0, 0.006, days))
        high = np.maximum(open_, close) * np.exp(np.abs(rng.normal(0, 0.007, days)))
        low = np.minimum(open_, close) * np.exp(-np.abs(rng.normal(0, 0.007, days)))
        volume = rng.normal(4_000_000, 600_000, days).round()
        index = pd.bdate_range(end=self.as_of, periods=days)
        return pd.DataFrame({"open": open_, "high": high, "low": low, "close": close, "volume": volume}, index=index)


provider = YFinanceProvider() if DATA_SOURCE == "yahoo" else SyntheticProvider(AS_OF)
symbol = SYMBOL.strip().upper()

all_expiries = provider.expirations(symbol)
if not all_expiries:
    raise CalculatorError(f"No options found for stock symbol '{symbol}'.")
print(f"{symbol}: {len(all_expiries)} listed expiries")
print([e.isoformat() for e in all_expiries[:12]], "..." if len(all_expiries) > 12 else "")

## 3. `select_expiries`: which expiries to look at

We only need the term structure from now out to 45 days. The function:

1. sorts the expiries;
2. keeps everything up to **and including** the first expiry that is at least `max_dte` (45) days away.
   That one extra point lets the curve be interpolated at exactly 45 DTE;
3. drops an expiry that falls *today*, since a 0-DTE option has almost no time value and its IV is noise.

If no expiry reaches 45 days, the term structure can't be built and the calculator stops.
Selecting first also means we only download the chains we need. Fetching chains is the slow, rate-limited part.

In [ ]:
def select_expiries(expiries, as_of, max_dte=45):
    cutoff = as_of + timedelta(days=max_dte)
    ordered = sorted(expiries)
    for i, expiry in enumerate(ordered):
        if expiry >= cutoff:
            selected = ordered[: i + 1]
            return selected[1:] if selected[0] == as_of else selected
    raise CalculatorError("Not enough option data.")


selected = select_expiries(all_expiries, AS_OF, params.max_dte)
pd.DataFrame({"expiry": selected, "dte": [(e - AS_OF).days for e in selected]})

## 4. Fetch the option chains and the spot price

One chain per selected expiry. Below is the start of the first (front) chain's calls, already normalised to `strike, iv, bid, ask`.

In [ ]:
chains = [provider.option_chain(symbol, e) for e in selected]

spot = provider.spot(symbol)
if spot is None:
    raise CalculatorError("Unable to retrieve underlying stock price.")

print(f"Spot price: {spot:,.2f}")
front_calls = chains[0].calls
front_calls.iloc[(front_calls["strike"] - spot).abs().argsort()[:7]].sort_values("strike")

## 5. Find the at-the-money (ATM) options

"At the money" means the strike closest to the current price. The calculator picks it **separately** for calls and puts
using `(strike - spot).abs().idxmin()`, the row with the smallest distance to spot.

ATM options matter because their IV is the market's cleanest estimate of overall volatility for that expiry.
Out-of-the-money strikes add skew effects.

In [ ]:
def atm_rows(chain, spot):
    call = chain.calls.loc[(chain.calls["strike"] - spot).abs().idxmin()]
    put = chain.puts.loc[(chain.puts["strike"] - spot).abs().idxmin()]
    return call, put


call, put = atm_rows(chains[0], spot)
pd.DataFrame({"call": call, "put": put}).T

## 6. `is_stale`: reject cleared quotes

This guard wasn't in the original calculator. It was added after Yahoo returned "cleared" chains overnight:

- **bid = 0 and ask = 0**: no market maker is quoting. A real ATM option always has an ask.
- **IV ≤ 0.0001**: Yahoo reports IV as 0.00001 when it can't compute one.

Without this guard, those placeholder IVs flow through the maths and the stock is silently scored **Avoid**,
with an absurd iv30/rv30 such as 0.003. With it, the calculator stops and explains why.

In [ ]:
STALE_IV_FLOOR = 1e-4


def is_stale(quote):
    return (quote["bid"] == 0 and quote["ask"] == 0) or float(quote["iv"]) <= STALE_IV_FLOOR


for chain in chains:
    c, p = atm_rows(chain, spot)
    if is_stale(c) or is_stale(p):
        raise CalculatorError(
            f"Stale option quotes for {chain.expiry.isoformat()}: ATM bid/ask are zero or IV is at the provider's floor. "
            "Yahoo clears option quotes overnight; try again after the market opens, or set DATA_SOURCE = 'synthetic'."
        )
print("All ATM quotes look live.")

## 7. ATM IV per expiry: the term structure points

For each expiry: **ATM IV = (call IV + put IV) / 2**. Averaging the two sides smooths out small call/put differences.

Each result becomes a `TermPoint(expiry, dte, atm_iv)`. Read together, these points are the **IV term structure**:
how implied volatility changes with time to expiry. Before earnings, the front expiries usually sit well above the later ones.

Chains with no calls or no puts are skipped.

In [ ]:
@dataclass(frozen=True)
class TermPoint:
    expiry: date
    dte: int
    atm_iv: float


points = []
for chain in chains:
    if chain.calls.empty or chain.puts.empty:
        continue
    c, p = atm_rows(chain, spot)
    points.append(TermPoint(chain.expiry, (chain.expiry - AS_OF).days, (float(c["iv"]) + float(p["iv"])) / 2))

if not points:
    raise CalculatorError("Could not determine ATM IV for any expiration dates.")
if len(points) < 2:
    raise CalculatorError("Not enough option data to build a term structure.")

pd.DataFrame(points)

## 8. `straddle`: the expected move

A **straddle** is one ATM call plus one ATM put. Its price is roughly what the market expects the stock to move,
in either direction, by that expiry. We use the **mid** price, (bid + ask) / 2, of each leg on the **front** expiry:

$$\text{expected move} = \frac{\text{call mid} + \text{put mid}}{\text{spot}}$$

If any of the four quotes is missing (NaN), there's no reliable price, so the result is `None` rather than `nan%`.

In [ ]:
def straddle_price(call, put):
    quotes = [call["bid"], call["ask"], put["bid"], put["ask"]]
    if any(q is None or pd.isna(q) for q in quotes):
        return None
    call_mid = (float(call["bid"]) + float(call["ask"])) / 2
    put_mid = (float(put["bid"]) + float(put["ask"])) / 2
    return (call_mid + put_mid) or None


front_chain = next(ch for ch in chains if ch.expiry == points[0].expiry)
straddle = straddle_price(*atm_rows(front_chain, spot))
expected_move = straddle / spot if straddle else None

print(f"Front expiry:  {points[0].expiry} ({points[0].dte} DTE)")
print(f"Straddle:      {straddle:,.2f}" if straddle else "Straddle:      n/a")
print(f"Expected move: {expected_move:.2%}" if expected_move else "Expected move: n/a")

## 9. `term_structure`: a continuous IV curve

Expiries fall on fixed dates, but the checks need IV at **exactly** 30 and 45 days.
`term_structure` turns the points into a function `curve(dte)` by **linear interpolation**: a straight line between neighbouring points.
Outside the range it holds the first or last value flat instead of extrapolating.

`np.interp` does both of these. The original code used `scipy.interpolate.interp1d` plus manual clamping, which gives the same result.

In [ ]:
def term_structure(dtes, ivs):
    dtes, ivs = np.asarray(dtes, float), np.asarray(ivs, float)
    order = dtes.argsort()
    dtes, ivs = dtes[order], ivs[order]
    return lambda dte: float(np.interp(dte, dtes, ivs))


curve = term_structure([p.dte for p in points], [p.atm_iv for p in points])

xs = np.linspace(0, points[-1].dte + 5, 200)
plt.figure(figsize=(8, 4))
plt.plot(xs, [curve(x) for x in xs], label="interpolated curve")
plt.scatter([p.dte for p in points], [p.atm_iv for p in points], color="black", zorder=3, label="ATM IV per expiry")
for d in (points[0].dte, params.window, params.max_dte):
    plt.axvline(d, color="grey", ls=":", lw=1)
    plt.annotate(f"{d}d: {curve(d):.3f}", (d, curve(d)), textcoords="offset points", xytext=(4, 6))
plt.xlabel("Days to expiry"); plt.ylabel("ATM implied vol"); plt.title(f"{symbol} IV term structure ({DATA_SOURCE} data)"); plt.legend()
plt.show()

## 10. Check 1, `ts_slope_0_45`: is the front month inflated?

The slope is the change in IV per day, measured between the front expiry and 45 DTE:

$$\text{slope} = \frac{IV(45) - IV(\text{front DTE})}{45 - \text{front DTE}}$$

A **negative** slope means front IV is higher than later IV (backwardation). That is the pre-earnings pattern a calendar spread needs.
The threshold of −0.00406 asks for a fairly steep drop: roughly 0.17 vol points over 41 days.

Each check is stored as a `Check(value, threshold, op, passed)`, so the result keeps the **number**, not just pass/fail.

In [ ]:
@dataclass(frozen=True)
class Check:
    value: float
    threshold: float
    op: Literal[">=", "<="]
    passed: bool


def at_least(value, threshold):
    return Check(value, threshold, ">=", value >= threshold)


def at_most(value, threshold):
    return Check(value, threshold, "<=", value <= threshold)


front = points[0]
if front.dte == params.max_dte:
    raise CalculatorError(f"Front expiry is exactly {params.max_dte} DTE; slope is undefined.")

ts_slope = (curve(params.max_dte) - curve(front.dte)) / (params.max_dte - front.dte)
ts_slope_check = at_most(ts_slope, params.max_ts_slope_0_45)
ts_slope_check

## 11. `yang_zhang`: realised volatility (RV30)

To decide whether options are expensive, we compare implied vol with how much the stock **actually** moved.
Yang-Zhang is a realised-volatility estimator that uses the full daily bar (open, high, low, close),
not just closes, so it uses more information per day than close-to-close volatility. It combines three variances over a rolling 30-day window:

| Component | Formula (per day, logs) | Captures |
|---|---|---|
| overnight $\sigma_o^2$ | $\ln(\text{open} / \text{prev close})^2$ | gaps between sessions, such as earnings gaps |
| close $\sigma_c^2$ | $\ln(\text{close} / \text{prev close})^2$ | day-to-day moves |
| Rogers-Satchell $\sigma_{rs}^2$ | $\ln\frac{H}{O}\ln\frac{H}{C} + \ln\frac{L}{O}\ln\frac{L}{C}$ | intraday range, robust to drift |

$$\sigma^2 = \sigma_o^2 + k\,\sigma_c^2 + (1-k)\,\sigma_{rs}^2, \quad k = \frac{0.34}{1.34 + \frac{n+1}{n-1}}$$

The result is annualised with $\sqrt{252}$ (trading days per year), so it's on the same scale as IV.

**Quirks kept from the original:** textbook Yang-Zhang uses *open-to-close* returns for the $k$ term, but this code uses
close-to-close. Each variance is also a plain sum of squares divided by $n-1$, without subtracting the mean.
The numbers are therefore a little different from a textbook implementation, but they match the original calculator.

In [ ]:
def yang_zhang(prices, window=30, trading_periods=252):
    log_ho = np.log(prices["high"] / prices["open"])
    log_lo = np.log(prices["low"] / prices["open"])
    log_co = np.log(prices["close"] / prices["open"])
    log_oc = np.log(prices["open"] / prices["close"].shift(1))   # overnight gap
    log_cc = np.log(prices["close"] / prices["close"].shift(1))  # close to close

    rs = log_ho * (log_ho - log_co) + log_lo * (log_lo - log_co)  # Rogers-Satchell

    scale = 1.0 / (window - 1.0)
    open_vol = (log_oc**2).rolling(window).sum() * scale
    close_vol = (log_cc**2).rolling(window).sum() * scale
    window_rs = rs.rolling(window).sum() * scale

    k = 0.34 / (1.34 + (window + 1) / (window - 1))
    result = np.sqrt(open_vol + k * close_vol + (1 - k) * window_rs) * np.sqrt(trading_periods)
    return result.dropna()


history = provider.price_history(symbol, params.history_period)
rv_series = yang_zhang(history, window=params.window)
if rv_series.empty:
    raise CalculatorError("Not enough price history.")

rv30 = float(rv_series.iloc[-1])
print(f"{len(history)} daily bars, {len(rv_series)} rolling RV values; latest RV30 = {rv30:.4f}")
history.tail()

## 12. Check 2, `iv30_rv30`: are options expensive?

**IV30** is the term-structure curve read at 30 days. **RV30** is the latest Yang-Zhang value from Step 11.

$$\frac{IV30}{RV30} \geq 1.25$$

This means the options price in at least 25% more volatility than the stock has recently delivered. That gives room for IV to fall after earnings.

In [ ]:
iv30 = curve(params.window)
iv30_rv30_check = at_least(iv30 / rv30, params.min_iv30_rv30)

print(f"IV30 = {iv30:.4f}   RV30 = {rv30:.4f}")
iv30_rv30_check

In [ ]:
plt.figure(figsize=(8, 3.5))
plt.plot(rv_series.index, rv_series.values, label="RV30 (Yang-Zhang, rolling)")
plt.axhline(iv30, color="tab:orange", ls="--", label=f"IV30 today = {iv30:.3f}")
plt.axhline(iv30 / params.min_iv30_rv30, color="grey", ls=":", label=f"max RV for a pass = {iv30 / params.min_iv30_rv30:.3f}")
plt.title(f"{symbol} realised vs implied volatility ({DATA_SOURCE} data)"); plt.legend()
plt.gcf().autofmt_xdate(); plt.show()

## 13. Check 3, `avg_volume`: is the stock liquid?

This is the 30-day rolling mean of daily share volume, using its latest value. Thinly traded stocks usually also have wide option spreads, which eat into a calendar's small edge.

In [ ]:
volume_series = history["volume"].rolling(params.window).mean().dropna()
if volume_series.empty:
    raise CalculatorError("Not enough price history.")

avg_volume = float(volume_series.iloc[-1])
avg_volume_check = at_least(avg_volume, params.min_avg_volume)
avg_volume_check

## 14. `recommend`: the verdict

| slope | volume | iv30/rv30 | verdict |
|---|---|---|---|
| ✅ | ✅ | ✅ | Recommended |
| ✅ | exactly one of the two ✅ | | Consider |
| anything else | | | Avoid |

The slope is mandatory. Without a front-month IV premium there is nothing for the calendar to capture.

In [ ]:
def recommend(avg_volume_ok, iv30_rv30_ok, ts_slope_ok):
    if avg_volume_ok and iv30_rv30_ok and ts_slope_ok:
        return "Recommended"
    if ts_slope_ok and avg_volume_ok != iv30_rv30_ok:
        return "Consider"
    return "Avoid"


recommendation = recommend(avg_volume_check.passed, iv30_rv30_check.passed, ts_slope_check.passed)
print(symbol)
print(recommendation)

## 15. Putting it together: `evaluate()`

Everything above, joined into one function with **structured input and output**:

- `Inputs` holds all the data the maths needs: symbol, date, spot, chains and price history. It has **no network access**.
- `evaluate(inputs, params) -> Result` is **pure**: the same inputs always give the same result. That makes it easy to test and lets you rerun it on saved data.
- `Result` keeps every value, not just the verdict, so it can be stored, compared or shown in any UI.
- `run(symbol)` is the convenience wrapper: fetch data through a provider, then `evaluate`.

This is the shape of `src/calculators/calendar_iv_crush.py` in chrysos-lab.

In [ ]:
@dataclass(frozen=True)
class Inputs:
    symbol: str
    as_of: date
    spot: float
    chains: list
    price_history: pd.DataFrame


@dataclass(frozen=True)
class Result:
    symbol: str
    as_of: date
    spot: float
    recommendation: str
    avg_volume: Check
    iv30_rv30: Check
    ts_slope_0_45: Check
    iv30: float
    rv30: float
    front_expiry: date
    straddle: float | None
    expected_move: float | None
    term_structure: list = field(default_factory=list)


def evaluate(inputs, params=Params()):
    wanted = set(select_expiries((c.expiry for c in inputs.chains), inputs.as_of, params.max_dte))
    chains = sorted((c for c in inputs.chains if c.expiry in wanted), key=lambda c: c.expiry)

    points, straddle = [], None
    for chain in chains:
        if chain.calls.empty or chain.puts.empty:
            continue
        call, put = atm_rows(chain, inputs.spot)
        if is_stale(call) or is_stale(put):
            raise CalculatorError(f"Stale option quotes for {chain.expiry.isoformat()}; Yahoo clears option quotes overnight.")
        points.append(TermPoint(chain.expiry, (chain.expiry - inputs.as_of).days, (float(call["iv"]) + float(put["iv"])) / 2))
        if len(points) == 1:
            straddle = straddle_price(call, put)

    if not points:
        raise CalculatorError("Could not determine ATM IV for any expiration dates.")
    if len(points) < 2:
        raise CalculatorError("Not enough option data to build a term structure.")
    front = points[0]
    if front.dte == params.max_dte:
        raise CalculatorError(f"Front expiry is exactly {params.max_dte} DTE; slope is undefined.")

    curve = term_structure([p.dte for p in points], [p.atm_iv for p in points])
    ts_slope = (curve(params.max_dte) - curve(front.dte)) / (params.max_dte - front.dte)

    rv_series = yang_zhang(inputs.price_history, window=params.window)
    volume_series = inputs.price_history["volume"].rolling(params.window).mean().dropna()
    if rv_series.empty or volume_series.empty:
        raise CalculatorError("Not enough price history.")
    iv30, rv30 = curve(params.window), float(rv_series.iloc[-1])

    vol_c = at_least(float(volume_series.iloc[-1]), params.min_avg_volume)
    ivrv_c = at_least(iv30 / rv30, params.min_iv30_rv30)
    slope_c = at_most(ts_slope, params.max_ts_slope_0_45)

    return Result(
        symbol=inputs.symbol, as_of=inputs.as_of, spot=inputs.spot,
        recommendation=recommend(vol_c.passed, ivrv_c.passed, slope_c.passed),
        avg_volume=vol_c, iv30_rv30=ivrv_c, ts_slope_0_45=slope_c,
        iv30=iv30, rv30=rv30, front_expiry=front.expiry,
        straddle=straddle, expected_move=straddle / inputs.spot if straddle else None,
        term_structure=points,
    )


def run(symbol, provider, params=Params(), as_of=None):
    symbol = symbol.strip().upper()
    if not symbol:
        raise CalculatorError("No stock symbol provided.")
    as_of = as_of or date.today()
    expiries = provider.expirations(symbol)
    if not expiries:
        raise CalculatorError(f"No options found for stock symbol '{symbol}'.")
    chains = [provider.option_chain(symbol, e) for e in select_expiries(expiries, as_of, params.max_dte)]
    spot = provider.spot(symbol)
    if spot is None:
        raise CalculatorError("Unable to retrieve underlying stock price.")
    history = provider.price_history(symbol, params.history_period)
    return evaluate(Inputs(symbol, as_of, spot, chains, history), params)

Run `evaluate` on the **same data** fetched above and confirm it matches the step-by-step values.

In [ ]:
result = evaluate(Inputs(symbol, AS_OF, spot, chains, history), params)

assert result.recommendation == recommendation
assert math.isclose(result.ts_slope_0_45.value, ts_slope_check.value)
assert math.isclose(result.iv30_rv30.value, iv30_rv30_check.value)
assert math.isclose(result.avg_volume.value, avg_volume_check.value)
assert result.expected_move == expected_move
print("evaluate() matches the step-by-step calculation.")

### Output table

This is the same layout as the chrysos-lab terminal command (`uv run calendar-iv-crush NFLX`).

In [ ]:
def format_result(r):
    def check_row(name, c, fmt):
        return (name, "PASS" if c.passed else "FAIL", fmt.format(c.value), f"{c.op} {fmt.format(c.threshold)}")

    rows = [
        check_row("avg_volume", r.avg_volume, "{:,.0f}"),
        check_row("iv30_rv30", r.iv30_rv30, "{:.3f}"),
        check_row("ts_slope_0_45", r.ts_slope_0_45, "{:.5f}"),
        ("iv30", "", f"{r.iv30:.4f}", ""),
        ("rv30", "", f"{r.rv30:.4f}", ""),
        ("spot", "", f"{r.spot:,.2f}", ""),
        ("straddle", "", "n/a" if r.straddle is None else f"{r.straddle:,.2f}", f"(front expiry {r.front_expiry})"),
        ("expected_move", "", "n/a" if r.expected_move is None else f"{r.expected_move:.2%}", ""),
    ]
    headers = ("Item", "Result", "Value", "Threshold")
    widths = [max(len(row[i]) for row in (headers, *rows)) for i in range(4)]
    join = lambda cells: "  ".join(c.ljust(w) for c, w in zip(cells, widths)).rstrip()
    lines = [r.symbol, r.recommendation, "", join(headers), join(tuple("-" * w for w in widths))]
    lines += [join(row) for row in rows]
    curve_text = " | ".join(f"{p.dte}d {p.atm_iv:.3f}" for p in r.term_structure)
    lines += ["", f"Term structure (DTE / ATM IV): {curve_text}", f"As of {r.as_of}"]
    return "\n".join(lines)


print(format_result(result))

## 16. Screen several tickers

Now that each piece is understood, `run()` does the whole thing in one call. Errors such as stale quotes or missing options
are caught per ticker, so one bad symbol doesn't stop the loop.

In [ ]:
TICKERS = ["AAPL", "NFLX", "TSLA", "AMD"] if DATA_SOURCE == "yahoo" else ["DEMO"]

rows = []
for t in TICKERS:
    try:
        r = run(t, provider, params, AS_OF)
        rows.append({"symbol": r.symbol, "verdict": r.recommendation,
                     "avg_volume": round(r.avg_volume.value), "iv30_rv30": round(r.iv30_rv30.value, 3),
                     "ts_slope_0_45": round(r.ts_slope_0_45.value, 5),
                     "expected_move": None if r.expected_move is None else f"{r.expected_move:.2%}"})
    except CalculatorError as e:
        rows.append({"symbol": t, "verdict": f"Error: {e}"})

pd.DataFrame(rows)

### Try it yourself

- Change `SYMBOL` at the top and use **Runtime → Run all**.
- Tighten or loosen a threshold, e.g. `params = Params(min_iv30_rv30=1.4)`, and rerun from Step 1.
- Set `DATA_SOURCE = "synthetic"` to explore the steps when US markets are closed.